# Solution B2 · Lasso and noise features: automatic selection and scale

**Type**: Implement with scikit-learn

## Objective

Add 30 pure noise features to the diabetes dataset, use `LassoCV` to see which ones survive, and find out
how the result changes if you forget the scaler.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_real, y = ds.diabetes()
rng = np.random.default_rng(0)
noise = pd.DataFrame(rng.normal(size=(len(X_real), 30)), columns=[f"noise_{i}" for i in range(30)])
X = pd.concat([X_real, noise], axis=1)          # 10 true features + 30 of pure noise
real_columns = list(X_real.columns)
X.shape

## Commented solution

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

def rmse(model, columns):
    return mean_squared_error(y_te, model.predict(X_te[columns]), squared=False)

ols_real = LinearRegression().fit(X_tr[real_columns], y_tr)
ols_all = LinearRegression().fit(X_tr, y_tr)
rmse_ols_real = rmse(ols_real, real_columns)
rmse_ols_all = rmse(ols_all, list(X.columns))
print(f"OLS on 10 true features: {rmse_ols_real:.2f} | OLS on all 40: {rmse_ols_all:.2f}")

In [ ]:
lasso_scaled = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0, max_iter=20000)).fit(X_tr, y_tr)
rmse_lasso = rmse(lasso_scaled, list(X.columns))
coef = lasso_scaled[-1].coef_
n_real_kept = int((coef[:10] != 0).sum())
n_noise_kept = int((coef[10:] != 0).sum())
print(f"Lasso with scaler: RMSE {rmse_lasso:.2f} | true kept {n_real_kept}/10 | noise kept {n_noise_kept}/30")

In [ ]:
lasso_raw = LassoCV(cv=5, random_state=0, max_iter=20000).fit(X_tr, y_tr)
coef_r = lasso_raw.coef_
n_real_kept_raw = int((coef_r[:10] != 0).sum())
n_noise_kept_raw = int((coef_r[10:] != 0).sum())
print(f"Lasso without scaler: true kept {n_real_kept_raw}/10 | noise kept {n_noise_kept_raw}/30 "
      f"| chosen alpha {lasso_raw.alpha_:.3f}")

In [ ]:
print("standard deviation of the true features :", X_tr[real_columns].std().mean().round(3))
print("standard deviation of the noise features:", X_tr.drop(columns=real_columns).std().mean().round(3))
answer = "b"

## Check

In [ ]:
with verify():
    check(rmse_ols_all > rmse_ols_real, "adding noise makes OLS worse",
          "OLS on 40 features should have a higher RMSE than the one on 10")
    check(rmse_lasso < rmse_ols_all, "the scaled Lasso beats OLS on the 40 features",
          "with scaler and Lasso you should do better than OLS on all the features")
    check(n_real_kept >= 3, "the Lasso keeps several true features",
          "the scaled Lasso should keep at least a few true features")
    check(n_noise_kept < 10, "the scaled Lasso discards most of the noise",
          "with the scaler the Lasso should discard almost all of the 30 noise features")
    check(n_noise_kept_raw >= 25, "without the scaler the Lasso keeps almost all the noise",
          "without the scaler almost all of the 30 noise features should survive: did you remove the StandardScaler?")
    check(n_real_kept_raw + n_noise_kept_raw > 0, "the raw model has been trained",
          "count the nonzero coefficients of lasso_raw.coef_")
    check(answer == "b", "correct explanation", "reread the standard deviation of the true features and of the noise")
done()

## What you learned

- The scaler makes the coefficients **comparable**, so the L1 penalty hits the features according to how much they
  matter and not to how they are measured. With the scaler the Lasso keeps 5 true features and 4 noise ones; without the scaler
  it keeps *all 30* noise features.
- The mismatch here is extreme (0.05 against 1), but in real data it happens all the time: an income in euros next to
  an age in years.
- The Lasso improves on OLS with 40 features but does not bring it to the level of OLS on the 10 true ones alone: automatic
  selection has a cost, because it chooses based on the data and does not know the truth.

## Common pitfall

Letting `LassoCV` choose `alpha` does not fix a scale problem: it chooses the best alpha **for those
features as they were**. The scaler goes in the pipeline (so it is fitted only on the training fold at every
round of cross-validation), not applied by hand beforehand.

## Interview question

> *Click on a question to show its answer.*

<details><summary>You used the Lasso to select features and only 6 out of 200 survived. Can you say that the other 194 don't matter?</summary>

No. The Lasso chooses based on that sample: with correlated features it keeps one per group arbitrarily and discards
the others, and the selection can change by repeating the fit on another bootstrap sample. I can say that those six
are enough, together, to predict well *in that dataset*; not that the others have no relationship with the target. For a
substantive conclusion I would check the stability of the selection or use an ElasticNet.

</details>